# Лекция: Компьютерное зрение
## Object Detection

**Преподаватель:** Весельев Александр, Т-Банк

**Аудитория:** ФКН ВШЭ, МФТИ

---

## Содержание

1. **Постановка задачи** 
2. **Классические датасеты задачи детекции** 
4. **Метрики задачи object detection**
3. **Two-Stage** 
 - RCNN, Fast-RCNN – историческая справка
 - Faster-RCNN – детали
5. **One-Stage**
 - SSD
 - YOLO
 - RetinaNet
 - CenterNet, CornerNet
 - FCOS
 - DETR

---

# Что такое Object Detection?

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 11


def visualize_bounding_boxes(
        image: np.ndarray | Image.Image,
        boxes: list[list[int]],
        labels: list[str],
        figsize: tuple[int, int] = (4, 8)
    ) -> None:
    if isinstance(image, Image.Image):
        image = np.array(image)

    _, ax = plt.subplots(1, figsize=figsize)
    ax.imshow(image)

    colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan']
    
    for i, (box, label) in enumerate(zip(boxes, labels)):
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1

        color = colors[i % len(colors)]

        rect = patches.Rectangle((x1, y1), width, height,
                                 linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)

        ax.text(x1, y1, label, color='white', fontsize=12,
                bbox=dict(facecolor=color, alpha=0.7, edgecolor='none', boxstyle="round,pad=0.3"))

    plt.axis('off')
    plt.show()

In [ ]:
CATS_PATH = "./assets/cats.jpg"
CATS_IMAGE = Image.open(CATS_PATH)
GT_CATS_BOXES_XYXY = [(220, 35, 820, 500), (90, 450, 960, 1050)]
GT_CATS_LABELS = ["cat", "cat"]

visualize_bounding_boxes(CATS_IMAGE, GT_CATS_BOXES_XYXY, GT_CATS_LABELS)

In [ ]:
def yolo_to_xyxy(x_center_norm, y_center_norm, width_norm, height_norm, img_w, img_h):
    x_center = x_center_norm * img_w
    y_center = y_center_norm * img_h
    width = width_norm * img_w
    height = height_norm * img_h

    x_min = x_center - width / 2
    y_min = y_center - height / 2
    x_max = x_center + width / 2
    y_max = y_center + height / 2

    return x_min, y_min, x_max, y_max

image = Image.open("assets/coco_example_image.jpg")

with open("assets/coco_example_image_annotation.txt") as f:
    annotations = f.read()
    classes = [int(x.split(" ")[0]) for x in annotations.strip().split("\n")]
    boxes_xywh = [tuple(map(float, x.split(" ")[1:])) for x in annotations.strip().split("\n")]

boxes_xyxy = [
    yolo_to_xyxy(*box, *image.size) for box in boxes_xywh
]

labels = ['horse',
 'horse',
 'person',
 'person',
 'person',
 'potted plant',
 'person',
 'person',
 'person'
]

In [ ]:
visualize_bounding_boxes(image, boxes_xyxy, labels, figsize=(8, 10))

## Boxes format

1. `XYXY` (xmin, ymin, xmax, ymax)
2. `XYWH` (xmin, ymin, width, height)
3. `CXCYWH` (center x, center y, width, height)
4. `Normalized CXCYWH` (YOLO format)

# Datasets

- [Pascal VOC](https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/)
  - ~11k изображений
  - 20 классов объектов
- [COCO](https://cocodataset.org/#home)
  - ~330k изображений
  - ~1.5 млн object instances
  - 80 классов объектов
  - сложные сцены (много объектов на одном изображении)
  - есть segmentation masks, keypoints, captions
- [Open Images Dataset](https://storage.googleapis.com/openimages/web/index.html)
  - ~9 млн изображений
  - 600+ классов
  - ~16 млн bounding boxes
- [ImageNet Detection](https://www.image-net.org/challenges/LSVRC/index.php)
  - ~450k bounding boxes
  - 200 классов
- [KITTI Vision Benchmark Suite](https://www.cvlibs.net/datasets/kitti/index.php)
  - ~15k изображений
  - классы:
    - car
    - pedestrian
    - cyclist
  - сцены с камер автомобиля
---

- [Набор датасетов](https://docs.ultralytics.com/datasets/detect/)

# Metrics


## IoU (Intersection Over Union)

[medium](https://medium.com/@helenjoy88/intersection-over-union-iou-evaluating-object-detection-a24e9441fed9)

In [ ]:
def xyxy_iou(a, b):
    def area(box):
        x1, y1, x2, y2 = box
        return max(0, x2 - x1) * max(0, y2 - y1)

    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih

    union = area(a) + area(b) - inter
    return inter / union if union > 0 else 0.0


def calculate_iou(
    gt_boxes: list[list[int]],
    pred_boxes: list[list[int]],
    gt_labels: list[int],
    pred_labels: list[int],
) -> tuple[list[list[float]], list[tuple[int, float]]]:
    """
    Returns:
        iou_matrix
        best_matches
    """
    n_gt = len(gt_boxes)
    n_pred = len(pred_boxes)

    iou_matrix = []
    best_matches = []

    for i in range(n_gt):
        row = []
        best_iou = -1.0
        best_j = -1

        for j in range(n_pred):
            if gt_labels[i] != pred_labels[j]:
                val = -1.0
            else:
                val = xyxy_iou(gt_boxes[i], pred_boxes[j])
                if val > best_iou:
                    best_iou = val
                    best_j = j

            row.append(val)

        if best_j == -1:
            best_matches.append((-1, -1))
        else:
            best_matches.append((best_j, best_iou))

        iou_matrix.append(row)

    return iou_matrix, best_matches

In [ ]:
CATS_IMAGE = Image.open("assets/cats.jpg")
GT_CATS_BOXES_XYXY = [(220, 35, 820, 500), (90, 450, 960, 1050)]
GT_CATS_LABELS = ["cat", "cat"]

pred_boxes_xyxy = [(200, 65, 800, 700), (190, 460, 1100, 1000)]

iou_mat, best_match = calculate_iou(GT_CATS_BOXES_XYXY, pred_boxes_xyxy, GT_CATS_LABELS, ["cat", "cat"])
print("IoU Matrix")
print(*iou_mat, sep="\n")
print("Best Match")
print(*[f"Best match idx: {idx} IoU: {iou}" for idx, iou in best_match], sep="\n")
visualize_bounding_boxes(CATS_IMAGE, GT_CATS_BOXES_XYXY + pred_boxes_xyxy, GT_CATS_LABELS + ["cat (Pred)"] * 2)

## Mean Average Precision (mAP)

[medium](https://jonathan-hui.medium.com/map-mean-average-precision-for-object-detection-45c121a31173)

Алгоритм вычисления:
1. Фиксируем IoU threshold, по которому определяем, true positive или false positive
2. Для каждого класса: сортируем предсказания по убыванию confidence
3. По отсортированному списку строим кривую Precision-Recall (PR-curve)
4. AP (Average Precision) = площадь под PR-кривой
5. mAP = среднее AP по всем классам

mAP – стандартная метрика, но не является [silver bullet](https://arxiv.org/pdf/1804.02767)

In [ ]:
def calculate_map(
    gt_boxes: list[list[int]],
    pred_boxes: list[list[int]],
    gt_labels: list[int],
    pred_labels: list[int],
    pred_conf: list[float],
    iou_thresh: float = 0.5,
    verbose: bool = False,
    interp_101: bool = False,
) -> tuple[float, dict[int, float]]:
    """
    Вычисляет mAP для задачи object detection.

    Для каждого класса:
      1. Сортируем предсказания по убыванию confidence.
      2. Жадно матчим каждое предсказание к ближайшему (по IoU) ещё не
         совпавшему GT-боксу того же класса.
      3. prediction → TP если IoU ≥ iou_thresh, иначе → FP.
         GT-бокс, не получивший предсказание, даёт FN (снижает recall).
      4. AP = площадь под PR-кривой.
      5. mAP = среднее AP по всем классам.

    Parameters
    ----------
    gt_boxes    : GT боксы в формате xyxy
    pred_boxes  : предсказанные боксы в формате xyxy
    gt_labels   : истинные метки классов
    pred_labels : предсказанные метки классов
    pred_conf   : уверенности предсказаний
    iou_thresh  : порог IoU для TP/FP
    verbose     : если True — строит PR-кривые для каждого класса
    interp_101  : если True — 101-point interpolation (COCO standard),
                  иначе — all-point interpolation (VOC 2010+)

    Returns
    -------
    mAP          : среднее AP по всем классам
    per_class_ap : {class_id -> AP}
    """
    classes = sorted(set(gt_labels) | set(pred_labels))

    per_class_ap: dict[int, float] = {}
    pr_data: dict[int, tuple] = {}

    for cls in classes:
        cls_gt = [gt_boxes[i] for i, l in enumerate(gt_labels) if l == cls]
        n_gt = len(cls_gt)

        cls_idx = sorted(
            [i for i, l in enumerate(pred_labels) if l == cls],
            key=lambda i: -pred_conf[i],
        )

        if not cls_idx or n_gt == 0:
            per_class_ap[cls] = 0.0
            continue

        cls_preds = [pred_boxes[i] for i in cls_idx]

        gt_matched = [False] * n_gt
        tp = np.zeros(len(cls_preds))
        fp = np.zeros(len(cls_preds))

        for p_idx, p_box in enumerate(cls_preds):
            best_iou, best_gt = 0.0, -1
            for g_idx, g_box in enumerate(cls_gt):
                if gt_matched[g_idx]:
                    continue
                iou = xyxy_iou(p_box, g_box)
                if iou > best_iou:
                    best_iou, best_gt = iou, g_idx

            if best_iou >= iou_thresh:
                tp[p_idx] = 1
                gt_matched[best_gt] = True
            else:
                fp[p_idx] = 1

        cum_tp = np.cumsum(tp)
        cum_fp = np.cumsum(fp)

        recalls = np.concatenate([[0.0], cum_tp / n_gt])
        precisions = np.concatenate([[1.0], cum_tp / (cum_tp + cum_fp)])

        # Envelope interpolation: P[i] = max(P[i], P[i+1], …, P[n])
        # Makes precision monotonically non-increasing (PASCAL VOC / COCO standard).
        for i in range(len(precisions) - 2, -1, -1):
            precisions[i] = max(precisions[i], precisions[i + 1])

        if interp_101:  # COCO standard
            recall_thresholds = np.linspace(0, 1, 101)
            inds = np.searchsorted(recalls, recall_thresholds, side='left')
            prec_at_thresh = np.array([
                precisions[pi] if pi < len(precisions) else 0.0
                for pi in inds
            ])
            ap = float(np.mean(prec_at_thresh))
        else:  # VOC 2010+
            ap = float(np.sum(np.diff(recalls) * precisions[1:]))

        per_class_ap[cls] = ap
        pr_data[cls] = (recalls, precisions, ap)

    if verbose and pr_data:
        n_cls = len(pr_data)
        fig, axes = plt.subplots(1, n_cls, figsize=(5 * n_cls, 4.5), squeeze=False)
        colors = plt.cm.Set1(np.linspace(0, 0.8, n_cls))

        for ax, (cls, (rec, prec, ap)), color in zip(axes[0], pr_data.items(), colors):
            ax.step(rec, prec, where="pre", color=color, lw=2, label=f"AP = {ap:.3f}")
            ax.fill_between(rec, prec, alpha=0.18, color=color, step="pre")
            ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
            ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
            ax.set_title(f"Class {cls}", fontweight="bold")
            ax.legend(loc="upper right", fontsize=9)
            ax.set_aspect("equal")

        mAP_val = float(np.mean(list(per_class_ap.values())))
        fig.suptitle(
            f"Precision-Recall Curves  |  mAP@{iou_thresh:.2f} = {mAP_val:.3f}",
            fontsize=13, fontweight="bold",
        )
        plt.tight_layout()
        plt.show()

    mAP = float(np.mean(list(per_class_ap.values()))) if per_class_ap else 0.0
    return mAP, per_class_ap

In [ ]:
CLASS_NAMES = {0: "cat", 1: "dog"}

gt_boxes = [
    [10,  10,  100, 100],
    [200, 10,  310, 110],
    [10,  200, 110, 310],
    [350, 10,  450, 110],
    [350, 150, 450, 260],
    [350, 300, 450, 400],
]
gt_labels = [0, 0, 0, 1, 1, 1]

pred_boxes = [
    [12,  12,  98,  98],
    [205, 12,  305, 112],
    [80,  100, 250, 220],
    [12,  205, 108, 305],
    [352, 12,  448, 108],
    [420, 80,  520, 190],
    [352, 155, 448, 255],
]
pred_labels = [0, 0, 0, 0, 1, 1, 1]
pred_conf   = [0.90, 0.75, 0.60, 0.45, 0.85, 0.65, 0.50]

map50, per_class_50 = calculate_map(
    gt_boxes, pred_boxes, gt_labels, pred_labels, pred_conf,
    iou_thresh=0.5,
    verbose=True,
)

print(f"mAP@0.5 = {map50:.4f}\n")
for cls, ap in per_class_50.items():
    print(f"  {CLASS_NAMES[cls]:>4s}  AP = {ap:.4f}")

In [ ]:
iou_thresholds = np.arange(0.50, 1.00, 0.05)

coco_per_class: dict[int, list[float]] = {0: [], 1: []}
coco_maps: list[float] = []

for thresh in iou_thresholds:
    m, pc = calculate_map(
        gt_boxes, pred_boxes, gt_labels, pred_labels, pred_conf,
        iou_thresh=thresh, verbose=False,
    )
    coco_maps.append(m)
    for cls in coco_per_class:
        coco_per_class[cls].append(pc.get(cls, 0.0))

coco_map = float(np.mean(coco_maps))

print("COCO-style mAP: разбивка по IoU-порогам")
print(f"{'IoU':>6}  {'mAP':>7}  {'AP cat':>8}  {'AP dog':>8}")
print("─" * 40)
for thresh, m, ap0, ap1 in zip(
    iou_thresholds, coco_maps, coco_per_class[0], coco_per_class[1]
):
    print(f"{thresh:6.2f}  {m:7.4f}  {ap0:8.4f}  {ap1:8.4f}")
print("─" * 40)
print(f"{'mean':>6}  {coco_map:7.4f}  "
      f"{np.mean(coco_per_class[0]):8.4f}  "
      f"{np.mean(coco_per_class[1]):8.4f}")
print(f"\nmAP@0.5\t= {map50:.4f}")
print(f"mAP@[.50:.95]\t= {coco_map:.4f}")

In [ ]:
# Конечно, не обязательно самостоятельно реализовывать mAP
# Можно использовать готовые библиотеки, например, pycocotools
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import io
import contextlib


def xyxy_to_xywh_coco(box):
    """[x1, y1, x2, y2] -> [x, y, w, h] (COCO format)"""
    x1, y1, x2, y2 = box
    return [float(x1), float(y1), float(x2 - x1), float(y2 - y1)]


# Для pycocotools нужно подготовить данные в формате COCO
coco_gt_dict = {
    "images": [{"id": 1, "width": 600, "height": 600}],
    "annotations": [],
    "categories": [
        {"id": 1, "name": "cat"},
        {"id": 2, "name": "dog"},
    ],
}

for i, (box, label) in enumerate(zip(gt_boxes, gt_labels)):
    bx = xyxy_to_xywh_coco(box)
    coco_gt_dict["annotations"].append({
        "id": i + 1, "image_id": 1, "category_id": label + 1,
        "bbox": bx, "area": bx[2] * bx[3], "iscrowd": 0,
    })

with contextlib.redirect_stdout(io.StringIO()):
    coco_gt_obj = COCO()
    coco_gt_obj.dataset = coco_gt_dict
    coco_gt_obj.createIndex()

coco_dt_list = []
for box, label, conf in zip(pred_boxes, pred_labels, pred_conf):
    bx = xyxy_to_xywh_coco(box)
    coco_dt_list.append({
        "image_id": 1, "category_id": label + 1,
        "bbox": bx, "score": float(conf),
    })

with contextlib.redirect_stdout(io.StringIO()):
    coco_dt_obj = coco_gt_obj.loadRes(coco_dt_list)

In [ ]:
coco_eval = COCOeval(coco_gt_obj, coco_dt_obj, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

# Two-Stage Detection

## 0. NMS (Non-Maximum Suppression)

In [ ]:
CATS_IMAGE = Image.open("assets/cats.jpg")
LOTS_OF_BOXES = [(200, 55, 920, 530), (250, 30, 720, 600), (220, 35, 820, 500), (100, 490, 1060, 1000), (150, 430, 900, 1090), (90, 450, 960, 1050)]
LOTS_OF_CONFS = [0.7, 0.8, 0.9, 0.7, 0.8, 0.9]
LOTS_OF_LABELS = ["cat"] * len(LOTS_OF_BOXES)

visualize_bounding_boxes(CATS_IMAGE, LOTS_OF_BOXES, LOTS_OF_LABELS)

In [ ]:
def nms_xyxy(
    boxes: list[list[int]],
    conf: list[float],
    conf_thr: float = 0.1,
    iou_thr: float = 0.5,
) -> list[list[int]]:
    if len(boxes) != len(conf):
        raise ValueError("boxes and conf must have the same length.")

    candidates = [
        (b, s) for b, s in zip(boxes, conf)
        if s >= conf_thr
    ]
    if not candidates:
        return []

    candidates.sort(key=lambda x: x[1], reverse=True)
    kept: list[list[int]] = []

    while candidates:
        best_box, _ = candidates.pop(0)
        kept.append(best_box)

        remaining = []
        for box, score in candidates:
            if xyxy_iou(best_box, box) <= iou_thr:
                remaining.append((box, score))

        candidates = remaining

    return kept

In [ ]:
boxes_after_nms = nms_xyxy(LOTS_OF_BOXES, LOTS_OF_CONFS)

visualize_bounding_boxes(CATS_IMAGE, boxes_after_nms, ["cat"] * len(boxes_after_nms))

#### Soft-NMS

[arxiv](https://arxiv.org/pdf/1704.04503) ([medium](https://medium.com/@juneta.tao/nms-vs-soft-nms-vs-weighted-box-fusion-for-object-detection-28d46828ac79))

## 1. Anchor Boxes

**Anchor boxes** - это заранее заданные прямоугольники разных размеров и соотношений сторон, которые размещаются по всей карте признаков и служат начальными гипотезами для объектов.
Нейросеть не ищет объект "с нуля" а корректирует один из anchor boxes, чтобы он точно совпал с объектом

[medium](https://medium.com/@abhishekjainindore24/anchor-boxes-in-object-detection-1a7831e7cadf)

[neerc.ifmo](https://neerc.ifmo.ru/wiki/index.php?title=%D0%97%D0%B0%D0%B4%D0%B0%D1%87%D0%B0_%D0%BD%D0%B0%D1%85%D0%BE%D0%B6%D0%B4%D0%B5%D0%BD%D0%B8%D1%8F_%D0%BE%D0%B1%D1%8A%D0%B5%D0%BA%D1%82%D0%BE%D0%B2_%D0%BD%D0%B0_%D0%B8%D0%B7%D0%BE%D0%B1%D1%80%D0%B0%D0%B6%D0%B5%D0%BD%D0%B8%D0%B8&mobileaction=toggle_view_desktop#Anchor_boxes)

In [ ]:
def wh_iou(box_wh, anchors_wh, eps=1e-9):
    """
    box_wh: (2,) array [w,h]
    anchors_wh: (K,2) array
    returns: (K,) IoU values
    """
    w, h = box_wh
    aw = anchors_wh[:, 0]
    ah = anchors_wh[:, 1]

    inter_w = np.minimum(w, aw)
    inter_h = np.minimum(h, ah)
    inter = inter_w * inter_h

    union = (w * h) + (aw * ah) - inter
    return inter / (union + eps)


def kmeans_anchors(boxes_wh, k=9, iters=50, seed=0):
    """
    boxes_wh: (N,2) widths/heights (positive)
    Returns: anchors_wh (k,2)
    """
    rng = np.random.default_rng(seed)
    N = boxes_wh.shape[0]

    anchors = boxes_wh[rng.choice(N, size=k, replace=False)].copy()

    for _ in range(iters):
        ious = np.stack([wh_iou(boxes_wh[i], anchors) for i in range(N)], axis=0)  # (N,k)
        assign = np.argmax(ious, axis=1)

        new_anchors = anchors.copy()
        for j in range(k):
            cluster = boxes_wh[assign == j]
            if len(cluster) == 0:
                new_anchors[j] = boxes_wh[rng.integers(0, N)]
            else:
                new_anchors[j] = np.median(cluster, axis=0)

        if np.allclose(new_anchors, anchors, rtol=1e-4, atol=1e-4):
            anchors = new_anchors
            break
        anchors = new_anchors

    return anchors

def generate_toy_boxes(n=2000, seed=42):
    rng = np.random.default_rng(seed)

    probs = np.array([0.45, 0.35, 0.2])
    comps = rng.choice(3, size=n, p=probs)

    boxes = np.zeros((n, 2), dtype=np.float32)

    idx = comps == 0
    boxes[idx, 0] = rng.lognormal(mean=np.log(28), sigma=0.35, size=idx.sum())
    boxes[idx, 1] = rng.lognormal(mean=np.log(32), sigma=0.35, size=idx.sum())

    idx = comps == 1
    boxes[idx, 0] = rng.lognormal(mean=np.log(80), sigma=0.30, size=idx.sum())
    boxes[idx, 1] = rng.lognormal(mean=np.log(70), sigma=0.30, size=idx.sum())

    idx = comps == 2
    boxes[idx, 0] = rng.lognormal(mean=np.log(180), sigma=0.25, size=idx.sum())
    boxes[idx, 1] = rng.lognormal(mean=np.log(160), sigma=0.25, size=idx.sum())

    ar = rng.lognormal(mean=0.0, sigma=0.35, size=n)
    boxes[:, 0] *= ar
    boxes[:, 1] /= ar

    boxes = np.clip(boxes, 4, 512)
    return boxes


def create_anchors_toy_example(n_boxes=2000, k=9, seed=42, kmeans_iters=60):
    boxes_wh = generate_toy_boxes(n=n_boxes, seed=seed)
    anchors_wh = kmeans_anchors(boxes_wh, k=k, iters=kmeans_iters, seed=seed + 1)
    return boxes_wh, anchors_wh


def plot_boxes_and_anchors(boxes_wh, anchors_wh, title="Toy boxes + K-means anchors"):
    plt.figure(figsize=(7, 6))
    plt.scatter(boxes_wh[:, 0], boxes_wh[:, 1], s=8, alpha=0.25, label="dataset boxes (w,h)")
    plt.scatter(anchors_wh[:, 0], anchors_wh[:, 1], s=200, marker="X", label="anchors")

    for i, (w, h) in enumerate(anchors_wh, start=1):
        plt.text(w, h, f"  A{i}\n  ({w:.0f},{h:.0f})", fontsize=9, va="center")

    plt.xlabel("width (px)")
    plt.ylabel("height (px)")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


boxes, anchors = create_anchors_toy_example(n_boxes=2000, k=9, seed=42)
plot_boxes_and_anchors(boxes, anchors)

## 2. RCNN (Историческая справка #1)

[arxiv](https://arxiv.org/pdf/1311.2524)

---

#### Компоненты R-CNN:

**1. Region Proposal (Selective Search)**

**Что:**

* Находит **region proposals**, где может находиться объект

**Как:**

* **Selective Search**:
  * сегментирует изображение
  * объединяет похожие сегменты
  * генерирует ~2000 возможных bounding boxes

**Итог:**

* Получаем список потенциальных регионов с объектами

**2. Warping Regions**

**Что:**

* Приводит все найденные регионы к **одному фиксированному размеру**

**Как:**

* Каждый region proposal:
  * вырезается из изображения
  * **масштабируется** (*224×224*)

**Итог:**

* Все регионы имеют одинаковый размер для обработки CNN

**3. CNN Feature Extraction**

**Что:**

* Извлекает features из каждого региона

**Как:**

* Каждый region проходит через предобученную CNN
* Берётся *feature vector* из последнего FC слоя.

**Итог:**

* Каждый регион превращается в *feature vector*

**4. Classification (SVM)**

**Что:**

* Определяет, к какому классу относится объект в регионе.

**Как:**

* Для каждого класса обучается отдельный **SVM**.

**Итог:**

* Регион классифицируется:
  * объект класса X
  * или background

**5. Bounding Box Regression**

**Что:**

* Улучшает точность координат bounding box

**Как:**

* Обучается **линейный регрессор**.
* Он:
  * получает CNN features
  * предсказывает *коррекцию координат* (dx, dy, dw, dh).

**Итог:**

* Bounding box становится более точным

**6. Non-Maximum Suppression (NMS)**

С NMS уже знакомы

---

## 3. Fast RCNN (Историческая справка #2)

[arxiv](https://arxiv.org/pdf/1504.08083)


---

#### компоненты Fast R-CNN


**1. Region Proposal (Selective Search)**

То же самое, что и в `RCNN`

**2. CNN Feature Map**

**Что:**

* Извлекает *feature map*

**Как:**

* Всё изображение **один раз** проходит через CNN
* Получаем общую feature map

**Итог:**

* Не нужно запускать CNN дляz каждого региона

**3. ROI Projection**

**Что:**

* Переводит bounding boxes в координаты feature map

**Как:**

* Учитывается `stride`
* Bounding boxes масштабируются под feature map
* *Важно* происходит ошибка округления

**Итог:**

* Каждому proposal соответствует регион на feature map

**4. RoI Pooling**

**Что:**

* Преобразует регион произвольного размера в **фиксированный размер feature map**.

**Как:**

* Region делится на **grid**
* В каждом блоке применяется **max pooling**
* *Важно* происходит ошибка округления (отбрасывается остаток)

**Итог:**

* Каждый ROI становится одного шейпа

**5. Fully Connected Layers**

**Что:**

* Преобразует ROI признаки в *feature vector*

**Как:**

* ROI pooling output проходит через FC слои

**Итог:**

* Получаем вектор признаков для региона


**6. Classification Head (Softmax)**

**Что:**

* Определяет класс объекта

**Как:**

* Один softmax classifier овместо множества SVM в R-CNN

**Итог:**

* Предсказывается:
  * класс объекта
  * вероятность

**7. Bounding Box Regression**

Аналогично как в R-CNN

**8. Non-Maximum Suppression (NMS)**

Аналогично как в R-CNN

---


## Faster R-CNN

[arxiv](https://arxiv.org/pdf/1506.01497)

#### компоненты Faster R-CNN

---

**1. CNN Backbone**

* Извлекает *feature map* всего изображения
* Изображение проходит через CNN:
* Этот feature map **используется и RPN, и детектором**

**2. Region Proposal Network (RPN)**

* Генерирует **region proposals (bounding boxes)**
  * Небольшая CNN на feature map, выдающая
      * objectness score
      * bbox regression

**3. Anchors**

* задают набор базовых bounding boxes разных размеров и пропорций.
* В каждой точке feature map размещается несколько anchors:

**4. Proposal Generation**

* превращает anchors в **region proposals**
* RPN предсказывает `dx, dy, dw, dh` и корректирует anchor
  * применяют bbox regression
  * фильтруют по objectness score

**5. Non-Maximum Suppression (NMS)**

* удаляет перекрывающиеся proposals

**6. RoI Pooling**

Аналогично Fast R-CNN

**7. Detection Head**

* Предсказывает:
  * класс объекта (softmax)
  * точные координаты bounding box (регрессия)

---

In [ ]:
import torch
import torchvision
from torchvision.utils import draw_bounding_boxes
from torchvision.transforms.functional import to_pil_image, to_tensor
import requests
import time

In [ ]:
# https://docs.pytorch.org/vision/0.12/models#object-detection-instance-segmentation-and-person-keypoint-detection
COCO_INSTANCE_CATEGORY_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]


device: torch.device = torch.device('cuda') if torch.cuda.is_available() else torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
device = torch.device('cpu')
print(f"Using: {device=}")

In [ ]:
def load_faster_rcnn_model(device: torch.device):
    weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
    model.eval().to(device)
    return model, weights


@torch.inference_mode()
def infer(model, image_tensor, device: torch.device):
    image_tensor = image_tensor.to(device)
    return model([image_tensor])


def draw_predictions_on_image(img_uint8: torch.Tensor, pred: dict[str, torch.Tensor], threshold: float = 0.5) -> torch.Tensor:
    boxes = pred["boxes"]
    labels = pred["labels"]
    scores = pred["scores"]

    keep = scores >= threshold
    boxes = boxes[keep]
    labels = labels[keep]
    scores = scores[keep]

    if boxes.numel() == 0:
        return img_uint8, []

    text_labels = []
    for lab, sc in zip(labels.tolist(), scores.tolist()):
        name = COCO_INSTANCE_CATEGORY_NAMES[lab] if lab < len(COCO_INSTANCE_CATEGORY_NAMES) else str(lab)
        text_labels.append(f"{name} {sc:.2f}")

    drawn = draw_bounding_boxes(
        img_uint8,
        boxes=boxes,
        labels=text_labels,
        width=5,
    )
    return drawn


In [ ]:
faster_rcnn_model, faster_rcnn_weights = load_faster_rcnn_model(device=device)

# you can chose your fighter
url = "http://farm4.staticflickr.com/3746/9895771396_45554292ba_z.jpg"
url = None

if url is None:
    image = Image.open(CATS_PATH)
else:
    image = Image.open(requests.get(url, stream=True).raw)

print(image.size)

In [ ]:
# mps doesn't work btw
image_tensor = faster_rcnn_weights.transforms()(image)
start = time.perf_counter()
infer_result = infer(faster_rcnn_model, image_tensor, device)
print(f"Time taken: {time.perf_counter() - start}")
print(f"Results are: {infer_result}")

image_tensor_w_boxes = draw_predictions_on_image(image_tensor, infer_result[0], threshold=0.5)
to_pil_image(image_tensor_w_boxes)

---

# One-Stage Detection

## 1. SSD: Single Shot MultiBox Detector (Историческая справка #3)

[arxiv](https://arxiv.org/pdf/1512.02325)

---

Главная идея SSD: **всё делается за один проход сети (single shot)**, без отдельных стадий генерации кандидатов.

---

#### компоненты SSD

**Backbone**

* В оригинальной SSD использовалась `VGG16`
  * получает изображение
  * превращает его в feature maps
  * промежуточные карты признаков также сохраняем

**anchor boxes**

* На каждой точке feature map SSD размещает anchor boxes

**Предсказания**

* Для anchor box сеть предсказывает:
  * класс объекта
  * bounding box offsets
  * confidence score

**Non-Maximum Suppression (NMS)**

---

#### Обучение SSD

Обучение на
* PASCAL VOC
* MS COCO

Моменты обучения, которые хотелось бы помнить:

* **Matching**
  * IoU для матчинга anchor box с ground truth
  * Важно:
    * Всегда есть 1 ответственный anchor box
    * Если IoU > 0.5 с другими, то matching может быть one-to-many

* **Loss**

  * Localization loss
    * Smooth L1 loss
  * Classification loss
    * softmax loss

* **Hard Negative Mining**

  * большинство boxes не несут информации:
```text
10000 boxes
9800 background
200 objects
```
  * Чтобы бороться с проблемой шумного сигнала, используются техники **hard negative mining**: производится downsample background boxes, чтобы их количество было примерно равно количеству объектов. [Почитать подробнее про hnm](https://medium.com/@sundardell955/hard-negative-mining-91b5792259c5)

---

In [ ]:
# In case u want
ssd_model = torchvision.models.detection.ssd300_vgg16(weights=torchvision.models.detection.SSD300_VGG16_Weights.DEFAULT)
ssd_model.to(device)
ssd_model.eval()

image_tensor = to_tensor(image)
start = time.perf_counter()
infer_result = infer(ssd_model, image_tensor, device)
print(f"Time taken: {time.perf_counter() - start}")

print(f"Results are: {infer_result}")

image_tensor_w_boxes = draw_predictions_on_image(image_tensor, infer_result[0], threshold=0.5)
to_pil_image(image_tensor_w_boxes)

## 2 YOLOv1 (историческая справка #4)

[arxiv](https://arxiv.org/pdf/1506.02640)

[сайт автора оригинальной модели](https://pjreddie.com/)

[оригинальный гитхаб](https://github.com/pjreddie/darknet)

---

Аналогично `SSD`, является single-shot object detector

---

**Общая идея YOLO v1**

* Сеть делит изображение на *сетку S × S*
* В каждой ячейке сетки может быть B bounding boxes (чьи центры попадают в ячейку)
* Каждая ячейка отвечает за **объекты, центр которых попадает в неё**

**Архитектура**

* CNN 
* FCN
  * 2 fully connected слоя после CNN


**Предсказание**

* Предсказывается S × S × (B × 5 + C) тензор:
  * S*S - сетка
  * B - количество bounding boxes в ячейке
  * 5 - координаты центра, ширина, высота, уверенность
  * C - количество классов

**NMS**

* Также используется NMS
* `confidence = confidence * P(class | object)`

---

**Проблемы**

* плохо с маленькими объектами
* только один класс на клетку
* нет anchor boxes


## 3 YOLOv3 (историческая справка #5)

[arxiv](https://arxiv.org/pdf/1804.02767)

---

Добавляются (в сравнении с `v1`)

1. anchor boxes
2. multi-scale detection
3. новый backbone (Darknet-53)
4. логистическую классификацию вместо softmax

---


## 4. Современные YOLO модели

Сейчас, семейство yolo – одно из самых популярных для решения задач object detection (и не только)

Последующее развитие было без оригинальных авторов и достигалось за счет большого количества инженерных трюков.

* [v4](https://arxiv.org/abs/2004.10934)
* [v5](https://arxiv.org/pdf/2407.20892)
* [v6](https://arxiv.org/abs/2209.02976)
* [v7](https://arxiv.org/abs/2207.02696)
* [v8](https://docs.ultralytics.com/ru/models/yolov8/)
* [v9](https://docs.ultralytics.com/ru/models/yolov9/)
* [v10](https://docs.ultralytics.com/ru/models/yolov10/)
* [v11](https://docs.ultralytics.com/ru/models/yolo11/)
* [v12](https://docs.ultralytics.com/ru/models/yolo12/)
* [v26](https://docs.ultralytics.com/models/yolo26/#overview)


Самая популярная компания, занимающаяся развитием серии моделей и выпуском новых версий, является [ultralytics](https://docs.ultralytics.com/) ([github](https://github.com/ultralytics/ultralytics))

---



In [ ]:
#!pip install ultralytics
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model(CATS_PATH)

annotated = results[0].plot()[..., ::-1]
plt.imshow(annotated)
plt.axis("off")
plt.show()

---

## 5. RetinaNet

[arxiv](https://arxiv.org/pdf/1708.02002)

---

**Главная идея: почему одностадийные детекторы трудно обучать**

В one-stage детекторах anchors очень много. Большинство неинформативны. Получается огромный дисбаланс:

* много легких отрицательных примеров
* мало положительных

**Focal Loss**

Focal Loss модифицирует бинарную кросс-энтропию так, чтобы:

* уменьшать вклад простых примеров (которые модель уже правильно классифицирует),
* усиливать вклад сложных

Для бинарного случая:

* пусть $p$ — предсказанная вероятность класса `1`
* $p_t = p$, если истинный класс 1; иначе $p_t = 1-p$

Тогда focal loss:
$$
FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)
$$

* $\gamma$ - `focusing parameter`
* $\alpha_t$ - баланс классов

In [ ]:
def focal_loss(pred: torch.Tensor, gt: torch.Tensor, alpha: float = 0.25, gamma: float = 2.) -> torch.Tensor:
    """
    pred - [bs, n_classes, h, w] (логиты)
    gt - [bs, 1, h, w] элементы в диапазоне [0 ... n_classes]  (0 = background)
    """
    bs, n_classes, h, w = pred.shape
    one_hot = torch.zeros((bs, n_classes + 1, h, w), dtype=torch.float32, device=pred.device)
    gt = gt.to(dtype=torch.int64, device=pred.device)
    one_hot.scatter_(dim=1, index=gt, src=torch.ones_like(gt, dtype=torch.float32, device=pred.device))
    one_hot = one_hot[:, 1:]

    p = torch.sigmoid(pred)
    pt = p * one_hot + (1.0 - p) * (1.0 - one_hot)
    log_pt = torch.log(pt.clamp(min=1e-8))

    return torch.sum(-alpha * (1.0 - pt) ** gamma * log_pt)

---

#### компоненты RetinaNet

**Backbone**

Обычно это ResNet (например, ResNet-50/101)

**FPN (Feature Pyramid Network)**

* FPN строит *пирамиду признаков*: несколько карт признаков на разных масштабах
* RetinaNet предсказывает объекты на каждом уровне пирамиды, чтобы покрывать разные размерыa


In [ ]:
from collections import OrderedDict


m = torchvision.ops.FeaturePyramidNetwork([10, 20, 30], 5)

x = OrderedDict()
x['feat0'] = torch.rand(1, 10, 64, 64)
x['feat2'] = torch.rand(1, 20, 16, 16)
x['feat3'] = torch.rand(1, 30, 8, 8)

output = m(x)
print([(k, v.shape) for k, v in output.items()])

**Anchor boxes**

* На каждой позиции каждой карты признаков задаётся несколько anchors, отличающихся:
  * масштабом (size)
  * отношением сторон (aspect ratio)

**Отдельные class & box subnets**
* RetinaNet использует две небольшие подсети которые применяются к каждому уровню FPN:
  * Classification subnet
    * выдаёт вероятность классов для каждого anchor
    * K вероятностей (по числу классов) для каждого anchor
    * сигмоида (multi-label style), а не softmax, чтобы каждая категория была независимой
  * Box regression subnet
    * выдаёт 4 числа (смещения) для каждого anchor

---

In [ ]:
head = torchvision.models.detection.retinanet.RetinaNetHead(256, 9, 91)


res = head([torch.rand(1, 256, 10, 10), torch.rand(1, 256, 20, 20), torch.rand(1, 256, 40, 40)])
# 10 * 10 * 9 + 20 * 20 * 9 + 40 * 40 * 9 = 18900
print(f"{res['cls_logits'].shape=} {res['bbox_regression'].shape=}")

**Как RetinaNet работает (инференс)**

* Backbone + FPN: получаем пирамиду признаков
* На каждом уровне для каждого anchor:
   * голова классификации даёт вероятности классов,
   * голова регрессии даёт корректировку рамки
* Декодируем рамки: применяем предсказанные смещения к anchor
* Фильтрация:
   * убираем предсказания с низкой уверенностью
   * применяем NMS

---

In [ ]:
retinanet = torchvision.models.detection.retinanet_resnet50_fpn_v2(weights=torchvision.models.detection.RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT)
retinanet.to(device)
retinanet.eval()

image_tensor = to_tensor(image)
start = time.perf_counter()
infer_result = infer(retinanet, image_tensor, device)
print(f"Time taken: {time.perf_counter() - start}")

print(f"Results are: {infer_result}")

image_tensor_w_boxes = draw_predictions_on_image(image_tensor, infer_result[0], threshold=0.5)
to_pil_image(image_tensor_w_boxes)

## 6. CenterNet

[arxiv](https://arxiv.org/pdf/1904.08189)

---

Главная идея: `представлять каждый объект как одну точку: его центр`

модель делает следующее:

1. строит heatmap центров объектов
2. определяет размер bounding box
3. уточняет точную позицию центра
4. объект определяется по одной точке: центру
5. Формально предсказывается:
$$
(center_x, center_y, width, height)
$$


Модель является *anchor-free*

---

#### Компоненты CenterNet

**Backbone**

* Чаще всего используются:
  * Hourglass Network
  * ResNet
  * DLA-34

* каждый пиксель feature map соответствует области исходного изображения

**Center Heatmap Head**

* Модель предсказывает heatmap центров объектов: `C × H × W`, где `C` - число классов, `H` и `W` - высота и ширина feature map.
* Каждый пиксель heatmap показывает:
$$
P(object\ center\ at\ (x,y))
$$

* *Как обучается heatmap*
  * Для каждого объекта:
    * его центр проецируется на feature map
    * вокруг него рисуется `Gaussian`
$$
Y_{xyc} = exp(-((x-x_c)^2+(y-y_c)^2)/(2\sigma^2))
$$

* Используется *focal loss*

**Size Head**

* делается отдельной регрессионной головой
* Используется `L1 loss`

**Offset Head**

* Тк feature map меньше оригинального изображения, то центр обьекта может быть между пикселями feature map
* Поэтому модель предсказывает **offset**: `(o_x, o_y)`
* offset исправляет квантование координат. Итоговый центр `x = (x_{heatmap} + o_x) * stride, y = (y_{heatmap} + o_y) * stride`
* Используется `L1 loss`

**Post-processing**

* Используется NMS
* Top-K центров (около 100)

---

**Проблемы**

* Сложно детектировать перекрывающиеся объекты
* Один центр = один объект

---

## 7.5 CornerNet

[arxiv](https://arxiv.org/pdf/1808.01244)

* Версия, которая предшествовала `CenterNet`
* Вместо центра предсказывались углы (top-left, bot-right)
* Новая операция – `CornerPooling`
* На постпроцессинге углы сопостовляются по близости эмбедингов
* В общем и целом, обучение сильно сложнее и менее стабильно

## 7. FCOS: Fully Convolutional One-Stage Object Detection

[arxiv](https://arxiv.org/pdf/1904.01355)

---

Главная идея: детекция объектов без anchor boxes. Модель предсказывает bounding box для каждой точки feature map

---

**Архитектура**

1. Backbone
2. Feature Pyramid Network (FPN)
3. Detection Heads
4. Centerness branch
5. Post-processing (NMS)

---

**Box regression branch**

* Для каждой точки модель предсказывает:
```
(l, t, r, b)
```

  * l — расстояние до левой границы bbox
  * t — расстояние до верхней границы
  * r — до правой
  * b — до нижней

**Classification branch**

* Предсказывает `class probabilities` через **focal loss**

**Centerness branch**

* оценивает, насколько точка находится близко к центру объекта
    * без centerness, точки возле краёв bbox дают плохие предсказания

* вычисляется:

$$
centerness =
\sqrt{
\frac{min(l,r)}{max(l,r)}
\times
\frac{min(t,b)}{max(t,b)}
}
$$

* во время инференса ` final_score = classification_score * centerness`

---

**Loss функции**


* *Classification – **Focal Loss**
* Box regression – **GIoU Loss**
* Centerness loss – Binary Cross Entropy

**Generalized IoU**

[arxiv](https://arxiv.org/pdf/1902.09630)

In [ ]:
def calc_giou_loss(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    Args:
        x (torch.Tensor): [n_boxes, 4], xyxy
        y (torch.Tensor): [n_boxes, 4], xyxy
    """
    eps = 1e-7

    x_area = (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    y_area = (y[:, 2] - y[:, 0]) * (y[:, 3] - y[:, 1])

    intersection_start = torch.maximum(x[:, :2], y[:, :2])
    intersection_end = torch.minimum(x[:, 2:], y[:, 2:])
    intersection_wh = (intersection_end - intersection_start).clip(min=0)
    intersection_area = intersection_wh[:, 0] * intersection_wh[:, 1]

    union_area = x_area + y_area - intersection_area
    iou = intersection_area / (union_area + eps)

    enclosing_box_start = torch.minimum(x[:, :2], y[:, :2])
    enclosing_box_end = torch.maximum(x[:, 2:], y[:, 2:])
    enclosing_box_wh = (enclosing_box_end - enclosing_box_start).clip(min=0)
    enclosing_box_area = enclosing_box_wh[:, 0] * enclosing_box_wh[:, 1] + eps

    giou_value = iou - (enclosing_box_area - union_area) / enclosing_box_area

    giou_loss = 1. - giou_value
    return torch.sum(giou_loss)

In [ ]:
# пример инференса

fcos = torchvision.models.detection.fcos_resnet50_fpn(weights=torchvision.models.detection.FCOS_ResNet50_FPN_Weights.DEFAULT)
fcos.to(device)
fcos.eval()

image_tensor = to_tensor(image)
start = time.perf_counter()
infer_result = infer(fcos, image_tensor, device)
print(f"Time taken: {time.perf_counter() - start}")

print(f"Results are: {infer_result}")

image_tensor_w_boxes = draw_predictions_on_image(image_tensor, infer_result[0], threshold=0.5)
to_pil_image(image_tensor_w_boxes)

## 8. DETR: DEtection TRansformer

[arxiv](https://arxiv.org/pdf/2005.12872)

---

* Решается задача detection как задачу set prediction
* Модель получает изображение и выдает фиксированное количество объектов
  * Каждый объект содержит `(class_label, bounding_box)`
  * Если объекта нет, модель предсказывает `no object`


---

#### Компоненты DETR

**CNN Backbone**

Обычно используется: ResNet-50 или ResNet-101

**Flatten + positional encoding**


* Feature map разворачивается:
  * Каждый элемент = *один spatial токен*
  * добавляется *позиционная информация*: `feature + position`

**Transformer Encoder**

* self-attention между всеми spatial токенами

**Object Queries**

* Вводится `N обучаемых векторов` (N = 100) - **object queries**
  * каждая query пытается "найти" один объект на изображении.

**Transformer Decoder**

* self-attention между queries
* cross-attention
    * Каждая query смотрит на encoder features

**Prediction heads**

* Каждая query проходит через MLP
  * MLP предсказывает класс (`softmax over classes + "no object"`), bounding bix (`(cx_center, cy_center, cwidth, cheight)`)
Она предсказывает:

**Hungarian Matching**

* [Почитать, что такое hungarian algorithm](https://en.wikipedia.org/wiki/Hungarian_algorithm)
* Используется только во время обучения
  * Hungarian matching делает оптимальное сопоставление
    * cost matrix состоит из:
      * classification loss
      * bbox l1 loss
      * GIoU loss
  * после сопоставления, unmatched query обучаются только на классификацию, matched на все

---

Модели на базе DETR активно развиваются:

* [DINO](https://github.com/IDEA-Research/DINO)
* [Deformable DETR](https://github.com/fundamentalvision/Deformable-DETR)
* [RT-DETR](https://docs.ultralytics.com/models/rtdetr/)
* [RF-DETR](https://github.com/roboflow/rf-detr) - *SOTA*

---


In [ ]:
# !pip install transformers
# !pip install timm
from transformers import DetrImageProcessor, DetrForObjectDetection
from transformers.models.detr.modeling_detr import DetrModelOutput, DetrObjectDetectionOutput

model_name = "facebook/detr-resnet-50"
local_files_only = True
processor = DetrImageProcessor.from_pretrained(model_name, local_files_only=local_files_only)
detr = DetrForObjectDetection.from_pretrained(model_name, local_files_only=local_files_only, dtype=torch.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
detr.to(device)
detr.eval()

inputs = processor(images=image, return_tensors="pt").to(device)

In [ ]:
with torch.no_grad():
    vision_features = detr.model.backbone(**inputs)
    feature_map, mask = vision_features[-1]

    projected_feature_map = detr.model.input_projection(feature_map)
    flattened_features = projected_feature_map.flatten(2).permute(0, 2, 1)
    spatial_position_embeddings = detr.model.position_embedding(
        shape=feature_map.shape, device=device, dtype=feature_map.dtype, mask=mask
    )
    flattened_mask = mask.flatten(1)

    encoder_outputs = detr.model.encoder(
        inputs_embeds=flattened_features,
        attention_mask=flattened_mask,
        spatial_position_embeddings=spatial_position_embeddings,
    )

    object_queries_position_embeddings = detr.model.query_position_embeddings.weight.unsqueeze(0).repeat(
        flattened_features.size(0), 1, 1
    )

    queries = torch.zeros_like(object_queries_position_embeddings)

    decoder_outputs = detr.model.decoder(
            inputs_embeds=queries,
            spatial_position_embeddings=spatial_position_embeddings,
            object_queries_position_embeddings=object_queries_position_embeddings,
            encoder_hidden_states=encoder_outputs.last_hidden_state,
            encoder_attention_mask=flattened_mask,
    )

    model_outputs = DetrModelOutput(
        last_hidden_state=decoder_outputs.last_hidden_state,
        decoder_hidden_states=decoder_outputs.hidden_states,
        decoder_attentions=decoder_outputs.attentions,
        cross_attentions=decoder_outputs.cross_attentions,
        encoder_last_hidden_state=encoder_outputs.last_hidden_state,
        encoder_hidden_states=encoder_outputs.hidden_states,
        encoder_attentions=encoder_outputs.attentions,
        intermediate_hidden_states=decoder_outputs.intermediate_hidden_states,
    )

    sequence_output = model_outputs[0]

    logits = detr.class_labels_classifier(sequence_output)
    pred_boxes = detr.bbox_predictor(sequence_output).sigmoid()

    outputs = DetrObjectDetectionOutput(
        loss=None,
        loss_dict=None,
        logits=logits,
        pred_boxes=pred_boxes,
        auxiliary_outputs=None,
        last_hidden_state=None,
        decoder_hidden_states=None,
        decoder_attentions=None,
        cross_attentions=None,
        encoder_last_hidden_state=None,
        encoder_hidden_states=None,
        encoder_attentions=None,
    )

results = processor.post_process_object_detection(
    outputs,
    target_sizes=torch.tensor([image.size[::-1]]).to(device),
    threshold=0.9
)[0]

COCO_INSTANCE_CATEGORY_NAMES
boxes = [box.cpu().numpy() for box in results["boxes"]]
labels = [f"{COCO_INSTANCE_CATEGORY_NAMES[label.item()]}: {score.item():.3f}" for label, score in zip(results["labels"], results["scores"])]


visualize_bounding_boxes(image=image, boxes=boxes, labels=labels)

---